# Common Failure Modes and How to Fix Them
Companion notebook for the **Common Failure Modes and How to Fix Them** section of Chapter 11.

Most production agent failures are visible if you log the right signals and reduce autonomy where needed -- bad routing, weak retrieval, wrong escalation, unsafe actions, and over-automation.

This is the book's `can_execute`, with one fix: the printed text has `return True` *indented inside* the `if` block, immediately after another `return` -- unreachable dead code, and every action *not* in the risky set falls through with no `return` at all (returning `None`, which is falsy but not an explicit, auditable "allowed"). Moved `return True` out to be the function's default for non-flagged actions:

In [1]:
def can_execute(action, ctx):
    if action in {"close_account", "change_payment", "cancel_visit"}:
        return ctx["identity_verified"] and ctx["user_confirmed"] and ctx["policy_pass"]
    return True

In [2]:
risky_ctx_incomplete = {"identity_verified": True, "user_confirmed": False, "policy_pass": True}
risky_ctx_complete = {"identity_verified": True, "user_confirmed": True, "policy_pass": True}

print("close_account, missing confirmation:", can_execute("close_account", risky_ctx_incomplete))
print("close_account, all checks passed:   ", can_execute("close_account", risky_ctx_complete))
print("read_order_status (not a risky action):", can_execute("read_order_status", {}))

close_account, missing confirmation: False
close_account, all checks passed:    True
read_order_status (not a risky action): True


**Bad routing / wrong tool choice**: detect it by logging intent, selected tool, and outcome for every run, then look for repeated mismatch patterns:

In [3]:
run_logs = [
    {"intent": "refund_question", "tool_called": "policy_lookup", "outcome": "ok"},
    {"intent": "refund_question", "tool_called": "billing_refund_tool", "outcome": "wrong_tool"},
    {"intent": "refund_question", "tool_called": "billing_refund_tool", "outcome": "wrong_tool"},
    {"intent": "check_order", "tool_called": "orders_api", "outcome": "ok"},
]


def find_routing_mismatches(logs: list[dict], min_occurrences: int = 2) -> dict:
    from collections import Counter
    mismatches = Counter(
        (log["intent"], log["tool_called"])
        for log in logs if log["outcome"] == "wrong_tool"
    )
    return {pair: count for pair, count in mismatches.items() if count >= min_occurrences}


repeated = find_routing_mismatches(run_logs)
print("repeated mismatch patterns (tighten routing rules for these):", repeated)

repeated mismatch patterns (tighten routing rules for these): {('refund_question', 'billing_refund_tool'): 2}


**Weak retrieval**: don't trust stale context for facts that change often -- prefer a live tool call when the retrieved item is too old:

In [4]:
from datetime import datetime, timedelta


def should_use_live_tool_instead(retrieved_at: datetime, freshness_limit_minutes: int = 5) -> bool:
    age = datetime.now() - retrieved_at
    return age > timedelta(minutes=freshness_limit_minutes)


stale_retrieval = datetime.now() - timedelta(minutes=20)
fresh_retrieval = datetime.now() - timedelta(minutes=1)

print("refund status retrieved 20 min ago -> call live tool instead:", should_use_live_tool_instead(stale_retrieval))
print("refund status retrieved 1 min ago -> retrieved text is fine:  ", should_use_live_tool_instead(fresh_retrieval))

refund status retrieved 20 min ago -> call live tool instead: True
refund status retrieved 1 min ago -> retrieved text is fine:   False


**Wrong escalation rate**: track it by intent so you can tell "escalates too little" from "escalates too much" apart, instead of tuning on gut feel:

In [5]:
def escalation_rate_by_intent(logs: list[dict]) -> dict:
    from collections import defaultdict
    totals, escalations = defaultdict(int), defaultdict(int)
    for log in logs:
        totals[log["intent"]] += 1
        if log.get("escalated"):
            escalations[log["intent"]] += 1
    return {intent: escalations[intent] / totals[intent] for intent in totals}


escalation_logs = [
    {"intent": "urgent_symptom", "escalated": True},
    {"intent": "urgent_symptom", "escalated": False},  # a miss -- this should have escalated
    {"intent": "reset_password", "escalated": True},   # over-escalated -- simple request
    {"intent": "reset_password", "escalated": False},
    {"intent": "reset_password", "escalated": False},
]

rates = escalation_rate_by_intent(escalation_logs)
for intent, rate in rates.items():
    print(f"{intent:16s} escalation_rate={rate:.0%}")

print("\nurgent_symptom at 50% is too low (lower the threshold);")
print("reset_password at 33% may be too high for a low-risk request (raise the threshold).")

urgent_symptom   escalation_rate=50%
reset_password   escalation_rate=33%

urgent_symptom at 50% is too low (lower the threshold);
reset_password at 33% may be too high for a low-risk request (raise the threshold).


**Over-automation**: if the same high-impact task keeps failing, don't just tune prompts -- reduce autonomy, moving from auto-execute to recommend-and-confirm, or from agent action to human approval:

In [6]:
def next_autonomy_level(current_level: str, recent_failure_count: int) -> str:
    levels = ["human_approval", "recommend_and_confirm", "auto_execute"]
    if recent_failure_count >= 3 and levels.index(current_level) > 0:
        return levels[levels.index(current_level) - 1]  # step down one level
    return current_level


print(next_autonomy_level("auto_execute", recent_failure_count=3))
print(next_autonomy_level("auto_execute", recent_failure_count=1))

recommend_and_confirm
auto_execute
